In [ ]:
## For colab
## Download the specific file to the current directory
!wget https://raw.githubusercontent.com/filanova/summer_school_2026/refs/heads/main/modules.py

# to delete a file execute:
# !rm modules.py

In [ ]:
## Download the file from GitLab
## -O renames the file locally to 'data.mat' so it's easy to refer to
!wget -O data.mat "https://gitlab.mpi-magdeburg.mpg.de/goyalp/learning_linear_stablesystems/-/raw/master/data/Burger_data_init_condions.mat"

## Verify the file was downloaded
#!ls -lh data.mat


In [ ]:
import torch                       # Import PyTorch
import torch.nn as nn              # Neural network library
from dataclasses import dataclass  # Standard library import for data objects
from scipy.io import loadmat       # Import function to load  .mat files
import numpy as np                 # Import NumPy
import matplotlib.pyplot as plt    # Standard library import for plots
import modules as mdl              # Specific library for this notebook (is provided in modules.py file)
from scipy.integrate import solve_ivp

---
# Stability-preserved operator learning

This tutorial shows  learning dynamical systems from data, with a specific focus on enforcing and analyzing asymptotic stability in linear and non-linear contexts.

---

## 1. Introduction to asymptotic stability
### 1.1. Continuous-time linear systems

Consider a continuous-time **Linear Time-Invariant (LTI)** system described by the state-space equation:

\begin{equation}
\tag{1}
\frac{d}{dt} \mathbf{x}(t) = \mathbf{A} \mathbf{x} (t), \quad \mathbf{x}(0) = \mathbf{x}_0,
\end{equation}


where  
* $\mathbf{x}(t) \in \mathbb{R}^{n} \quad \;$        - state variable,
* $\mathbf{x}_0 \in \mathbb{R}^{n} \quad \quad$         - initial condition,
* $\mathbf{A} \in \mathbb{R}^{n \times n} \quad $  - system matrix.

Dimension $n$ reflects the number of discretization points in the solution domain, called **degrees of freedom (DOFs)**.

We assume that snapshots of the state $\mathbf{x}(t)$ and the state derivatives $\frac{d}{dt} \mathbf{x}(t)$ are available for selected time points $t_1 , \ldots , t_k$, and collected in the snapshot matrices as follows:


$$
\mathbf{X} = \begin{bmatrix}
| & | & & | \\
\mathbf{x}(t_0) & \mathbf{x}(t_1) & \dots & \mathbf{x}(t_k) \\
| & | & & |
\end{bmatrix}, \quad
\dot{\mathbf{X}} = \begin{bmatrix}
| & | & & | \\
\frac{d}{dt}\mathbf{x}(t_0) & \frac{d}{dt}\mathbf{x}(t_1) & \dots & \frac{d}{dt}\mathbf{x}(t_k) \\
| & | & & |
\end{bmatrix}
$$


<u>Load data</u>

* The next cell loads the data from `Burger_data_init_condions.mat`.

* The mat-file contains solution snapshots `X_data` for [Burgers equation](https://en.wikipedia.org/wiki/Burgers%27_equation) for different initial conditions `xo`. Solution for all  initial conditions is performed at equidistant time points, which are stored in the vector `t`.

* After uploading the data to our notebook, we to define the variables that characterize the given data set (dimensions, time step, etc.).


In [ ]:
# ====================================================================================
# Load data
# ====================================================================================
data = loadmat("data.mat")                                    # For colab
t     = data["t"].T                                           # Time vector [n_times x 1]
X_all = data["X_data"].transpose(0, 2, 1)                     # Snapshots [n_inits x n_dofs x n_times]

n_inits, n_dofs, n_times  = X_all.shape                        # Assign the repective dimensions: number of initial conditions, number of time points, number of DOFs

dt = (t[2] - t[1]).item()                                     # Calculate the time step from the time vector

<u>Visualization</u>

* The next cell plots different trajctories using the predifined functions from `modules.py`.

    * `visualization1d([t_point1, t_point2, ...], time_step, snapshot, **kwargs)` plots the trajectories from one snapshot set at time points specified in `[t_point1, t_point2, ...]`. The trajectories `x` are the solution values at spatial domain coordinates `ξ`.

    * `visualization2d(tspan, snapshot, **kwargs)` renders the solution trajectories as a **2D** heatmap. This visualization captures the complete spatiotemporal dynamics, where the coordinates define the time `t` and spatial domain `ξ`, and the color intensity corresponds to the amplitude of the state variable `x`.

    * `visualization3d(tspan, snapshot, **kwargs)` renders the solution trajectories as a **3D** surface plot. This visualization captures the spatiotemporal evolution by mapping the time t and spatial domain `ξ` to the base axes, while the vertical axis represents the amplitude of the state variable `x`. This perspective allows for a detailed topological analysis of the system's stability and transient behavior.

In [ ]:
# ====================================================================================
# Visualization 1D
# ====================================================================================
mdl.visualize1d( [0.1, 0.2, 0.5], dt, X_all[0], title="Initial condition 0")  # here we plot three trajectories at t = 0.1s , t = 0.2s, and t = 0.5s
                                                                              # for the initial condition 0

#  plot here the trajectories at time points t = 0.0s and t = 0.9s for the initial condition 1

In [ ]:
# ====================================================================================
# Visualization 2D
# ====================================================================================
mdl.visualize2d(t, X_all[0], title="2D-plot over the domain. \n Initial condition 0.") # here we plot trajectories for time t for initial condition 0

# plot trajectories for initial condition 3

In [ ]:
# ====================================================================================
# Visualization 3D
# ====================================================================================

mdl.visualize3d(t, X_all[0], title="Initial condition 0")


### 1.2. Asymptotic stability

<u>**Definitions.**</u>

- **Lyapunov stability** : if for every $\epsilon > 0$, there exists a $\delta > 0$ such that if $\|\mathbf{x}(0) - x_e\| < \delta$, then $\|\mathbf{x}(t) - x_e \| < \epsilon$ for all $t \geq 0$. We call $x_e$ an equilibrium point.

- The equilibrium is called **assymptotically stable** if it is **Lyapunov stable**  and there exist $\delta > 0$ such that $$\text{if} \quad \|\mathbf{x}(0) - x_e\| < \delta, \quad \text{then:} \quad \lim_{t \to \infty} \|\mathbf{x}(t) - x_e\| = 0$$


- For linear systems, asymptotic stability is always **global**: it applies to **any** initial state $\mathbf{x}(0)$.



<u>**Conditions.**</u>

- A necessary and sufficient condition for the LTI system $$\frac{d}{dt} \mathbf{x}(t) = \mathbf{A} \mathbf{x} (t), \quad \mathbf{x}(0) = \mathbf{x}_0, $$
to be **asymptotically stable** is determined by the **eigenvalues** of the matrix $\mathbf{A}$. The system is asymptotically stable if and only if **all eigenvalues of $\mathbf{A}$ have strictly negative real parts**: $$\text{Re}(\lambda_j) < 0 \quad \forall j = 1, \dots, n ,$$
where $\lambda_j$ are the eigenvalues of $\mathbf{A}$. Matrix $\mathbf{A}$ is called **Hurwitz-stable**. This implies that all roots of the characteristic equation lie strictly in the open **left-half of the complex plane**.

- There exist a stability-preserving parametrizaion of the system matrix, as follows from the theorem:
> **Theorem:** Matrix $\mathbf{A} \in \mathbb{R}^{n \times n}$ is stable (Hurwitz, Lyapunov stable) if and only if it can be represented as
>
> $$\mathbf{A} = (\mathbf{J} - \mathbf{R})\mathbf{Q}$$
>
> where $\mathbf{J} = -\mathbf{J}^\top$ and $\mathbf{R} = \mathbf{R}^\top, \mathbf{Q} = \mathbf{Q}^\top$ are both positive definite.

<font color='blue'>**Your task:**</font>

* <font color='blue'>**complete the theorem proof and answer the questions.**</font>


> <u>**Proof**</u>
>
> We prove the equivalence by showing both implications: sufficiency ($\Leftarrow$) and necessity ($\Rightarrow$).
>
> **Sufficiency**($\Leftarrow$)
>
> Prove that if $\mathbf{A} = (\mathbf{J} - \mathbf{R})\mathbf{Q}$ with the properties defined above $\Rightarrow$ $\mathbf{A}$ is Hurwitz stable.
>
> Use the properties of $\mathbf{J},\mathbf{R},\mathbf{Q}$ to prove that the eigenvalues of $\mathbf{A}$ have negative real part.
>
>1. Write the fundamental equation for the matrix $\mathbf{A}$, eigenvalue $\lambda$, and eigenvector $\mathbf{z}$: $\mathbf{A}$ it holds: $$ \mathbf{A} ? = ? ?.$$
>2. Plug in the parametrization:
>$$
\mathbf{A} \mathbf{z} = ??? \mathbf{z} =  $$
> Replace $\tilde{\mathbf{z}} = \mathbf{Q} \mathbf{z} $
>$$
??? = ??? \tilde{\mathbf{z}} $$
> Multiply the equation with $\tilde{\mathbf{z}}^{H} $
>$$
\tilde{\mathbf{z}}^{H} (\mathbf{J} - \mathbf{R}) \tilde{\mathbf{z}} = ???? $$
> Rewrite the equation, open the brackets.
> Note that for skew-symmetric matrices real part of eigenvalues are zero; for SPD matrices the eigenvalues are real and positiv.
> $$ \underbrace{???}_{\text{purely imaginary}} - \underbrace{????}_{real, > 0} = \lambda \underbrace{???}_{real, >0} $$
> Write the eigenvalues of $\mathbf{J}$ as $i \gamma$ ($i$ - imaginary unit) , eigenvalues of $\mathbf{R}$ as $\beta$ , eigenvalues of $\mathbf{Q}$ as $\alpha$.
> Express $\lambda$:
> $$
\lambda =  $$
>
> What can you say about $Re(\lambda)$ ?.
>
> Q.E.D.
>
> **Necessity** ($\Rightarrow$)
>
> Prove that if $\mathbf{A}$ is Hurwitz stable $\Rightarrow$ we can construct matrices $\mathbf{J}, \mathbf{R}, \mathbf{Q}$ with above mentioned properties.
>
>1.  Since $\mathbf{A}$ is stable,  there exists a unique symmetric positive definite $\mathbf{P}$ such that:
>    $$\mathbf{A} \mathbf{P} + \mathbf{P} \mathbf{A}^\top < 0$$
>2.  **Construction of Q:**
>   Let $\mathbf{Q} = \mathbf{P}^{-1}$. Since $\mathbf{P} \succ 0$, then $\mathbf{Q} \succ 0$.
>3.  **Construction of J and R:**
>    Decompose the term $\mathbf{AP}$ into its symmetric and skew-symmetric parts:
>    $$\mathbf{AP} = \underbrace{\frac{\mathbf{AP} + \mathbf{P} \mathbf{A}^\top}{2}}_{\text{Symmetric}} + \underbrace{\frac{\mathbf{AP} - \mathbf{P A}^\top}{2}}_{\text{Skew-symmetric}}$$
>    * Define $\mathbf{J}$ as the skew-symmetric part:
>        $$\mathbf{J} = \frac{\mathbf{AP} - \mathbf{P A}^\top}{2}$$
> Is $\mathbf{J}$ skew symmetric? Check it.
>
>    * Define $\mathbf{R}$ as the symmetric and positive definite part:
>        $$\mathbf{R} = \frac{- \mathbf{AP} - \mathbf{P A}^\top}{2}$$
> Prove that $\mathbf{R}$ is SPD!
>
>**Q.E.D.**


## 2. Stability-Informed Learning Framework

- As mentioned above, our main assumption is that we only have simulation data and the system matrices are unknown ($\mathbf{A}$ is unknown). Additionally, our data is high-dimensional (number of DOFs $n$ is very large), so instead we want to have a system with

$$
\boxed{\text{Our goal is to learn the reduced linear operator } \hat{\mathbf{A}} \in \mathbb{R}^{r \times r} \text{ with guaranteed stability.}}
$$

- **The procedure simply contains two steps:**

    **1. Data compression,**

    **2. Optimization problem for the unknown $\hat{\mathbf{A}}$ with stability-preserving parametrization.**

- There are only two main steps, but they include many smaller tasks:

    1. Which data should be used in the optimization proces?
    2. How to reduce the data set? How to choose the reduced order $r$?
    3. How to formulate the optimization problem? Which initial values should be assigned to the unknown variables? What influence have learning hyperparameters on the results?
    4. How to evaluate the quality of the resulting reduced model?
    and so on ...

    <font color = blue>**Your task**</font>
    - <font color = blue>**suggest more questions!** </font>

<u>**Data for training and testing**</u>


- It is standard practice to split the available data into multiple sets to ensure robust performance:

    * **Training Set:** Used to optimize the model parameters.
    * **Test Set:** Used to evaluate the model on unseen data (not used during optimization).

<u>Training data</u>

In this example, we have multiple data sets, corresponding to different initial conditions. In the following cell, we divide the data into a training and test set.

In [ ]:
# ====================================================================================
# Training data
# ====================================================================================

all_idxs   = list(np.arange(0, n_inits))           # List of all indices
test_idxs  = list([4, 8, 12])                      # Data sets for test (e.g., data set corresponding to initial condition 4, to initial condition 8, and 12)
train_idxs = list(set(all_idxs) - set(test_idxs))  # Training indices

X_test     = X_all[test_idxs]                      # Include the data sets with indices test_idxs to the test set
X_training = X_all[train_idxs]                     # Include the data sets with indices training_idxs to the training set

n_inits_train = X_training.shape[0]                # Number of the training indices

print(f"Training trajectories: {X_training.shape[0]}")
print(f"Test trajectories: {X_test.shape[0]}")

<span style="color:deepskyblue; text-decoration:underline;">**Data compression: Low-dimensional representation**</span>

- Singular value decomposition (SVD) of a matrix $\mathbf{X} \in \mathbb{R}^{n \times k}$ is defined as $$\mathbf{X} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^{\top}$$ where
    * $\mathbf{U} \in \mathbb{R}^{n \times n}$ - orthogonal matrix ($\mathbf{U} \mathbf{U}^\top = \mathbf{U}^\top \mathbf{U} = \mathbf{I}$) containing left singular vectors (spatial modes),
    * $\mathbf{\Sigma} \in \mathbb{R}^{n \times k}$ - diagonal matrix with singular values $\sigma_{i+1} >= \sigma_i$,
    * $\mathbf{V} \in \mathbb{R}^{k \times k}$ - orthogonal matrix ($\mathbf{V} \mathbf{V}^T = \mathbf{V}^\top \mathbf{V} = \mathbf{I}$) containing right singular vectors (temporal modes).

- By using the subspace spanned by the *r* most dominant left singular vectors $\mathbf{V}_r$, we can obtain a <span style="color:deepskyblue; font-family:monospace;">low dimensional data representation </span>:$$ \mathbf{X}_r = \mathbf{U}_r^\top \mathbf{X},$$
where $\mathbf{X}_r \in \mathbb{R}^{r \times k}$.

- The information loss can be measured as $$\| \mathbf{X} - \mathbf{U}_r \mathbf{X}_r \|_2 \leq \sum_{i = r+1}^{n} \sigma_i $$

- There are different criteria for choosing the reduced dimension $r$, for example:

    * **Elbow-method** :The singular value decay is plotted in a semilogarithmic scale. The sharp bend of the curve - an "*elbow -point*" - indicates that from that dimension, the singular values have less contribution to the data and can be neglected without significant loss of the information.

    * **Cumulative energy method** : For each singular value cumulative energy is calculated: the ratio of the cumulative sum of the squared singular values to the total sum of the squared singular values.$$E_{\text{cum}} = \frac{\sum_{i=1}^{k} \sigma_i^2}{\sum_{i=1}^{n} \sigma_i^2}$$ This ratio represents the proportion of the data variance that is captured by the selected singular values. The energy method requires setting a threshold for the desired level of variance, such as 90% or 95%, and choosing the smallest number of singular values that satisfy this threshold.

    * and [many other](https://www.sciencedirect.com/science/article/pii/S2772415822000244?via%3Dihub) criteria.

<font color = blue>**Your task:**</font>


* <font color = blue>**perform an SVD decomposition for one data set**</font>
* <font color = blue>**choose a reduced order according to elbow or cumulative energy criteria**</font>
* <font color = blue>**analyze the data approximation error for diferent data sets**</font>


In [ ]:
# ===================================================================================================================
# Singular value decomposition
# ===================================================================================================================
U, S, V = ???  # use np.linalg.svd for a data set!


# Cumulative energy
Ecum = ??                                  # hint: use np.cumsum and np.sum functions!

r = ??                                     # specify the truncation order
mdl.plot_svd(r,S,Ecum)                     # Plot the singular value decay and cumulative energy
                                           # Horizontal line denotes a hypothetical truncation order r

In [ ]:
# ===================================================================================================================
# Cumulative energy criteria to choose the reduced order
# ===================================================================================================================
tol = 0.0001                         # Energy tolerance: small threshold, denoting how much energy we loose maximally
r_cum = np.argmax(Ecum > (1 - tol))  # Finds the first index where the condition is True.
                                     # Returns 0 if the condition is never met (so check .any() if unsure)

print(f"Order of reduced model: {r_cum}")
print(f"Energy captured by the snapshots: {100*Ecum[r_cum]}%")                 # Cumulative energy

In [ ]:
# ===================================================================================================================
# Choose the reduced order
# ===================================================================================================================
sys_order = ???

* <font color = blue>**Is $\mathbf{U}_r \mathbf{U}_r^\top = 1$ ?**</font>
* <font color = blue>**How to check the data reconstruction (approximation error)?**</font>

In [ ]:
# ===================================================================================================================
# Data approximation error
# ===================================================================================================================
Ur = U[:,:sys_order]
data_approx_error = ???
mdl.visualize2d(t, data_approx_error,
                contour=True,logscale=True,
                title = 'Data approximation error')

<span style="color:deepskyblue; text-decoration:underline;">**Data compression: Global low-dimensional basis**</span>

- Sometimes, it is better to extract a single and universal set of basis modes that capture the dynamics across *multiple* datasets (e.g., different experimental runs at various initial conditions).

- Instead of computing modes for a single dataset, we combine multiple datasets into one large matrix. This forces the resulting modes to be optimal for the entire range of operating conditions, making them ideal for **interpolation** or **reduced-order modeling (ROM)** across parameter spaces.

- Given snapshot matrices $\mathbf{X}_1, \mathbf{X}_2, \dots, \mathbf{X}_k$ corresponding to different parameters, we concatenate them **horizontally**:

$$
\mathbf{X}_{\text{global}} = \big[ \mathbf{X}_1 \;|\; \mathbf{X}_2 \;|\; \dots \;|\; \mathbf{X}_k \big]
$$
In our case, `k` = number of initial conditions `n_init` .

<font color = blue>**What is the dimension of the global snapshot matrix?**</font>
* **Rows:**
* **Columns:**  

In [ ]:
# ===================================================================================================================
# Global reduction basis
# ===================================================================================================================
#
#  Stack the data sets horizontally .
#  Concept: n_init different "Initial Conditions" (Sheets)
#
#           [ Init.Condition 1 ]   [ Init. Condition 2 ]     [ Init. Condition n_init ]
#                (Sheet 1)             (Sheet 2)                  (Sheet Ni)
#              <---- K ---->         <---- K ---->              <---- K ---->
#              +-----------+         +-----------+              +-----------+
#            ^ |           |         |           |              |           |
#     n_dofs | |  x^(1)    |         |  x^(2)    |     ...      |x^(n_init) |
#            v |           |         |           |              |           |
#              +-----------+         +-----------+              +-----------+
#
#
#
#  Output tensor has dimensions: (n_dofs, n_inits * n_times)
#  The time steps n_times are shared. The conditions are laid out side-by-side.
#
#            <-------------------- Ni * K ---------------------->
#              +-----------+-----------+ ... +-----------+
#            ^ |           |           |     |           |
#          M | |  x^(1)    |  x^(2)    | ... |x^(n_init) |
#            v |           |           |     |           |
#              +-----------+-----------+ ... +-----------+
#                ^           ^                 ^
#                |           |                 |
#              From        From              From
#           Init. Cond 1   Init. Cond 2      Init. Cond n_init


X_stack = X_training.transpose(1, 0, 2).reshape(X_training.shape[1], -1) # Stack the data set for each initial condition into one array

U_stack,S_stack,W_stack =   np.linalg.svd(X_stack, full_matrices = False)# use np.linalg.svd for the stacked data set!

squared_S_stack = S_stack**2                                             # Cumulative energy
Ecum_stack = np.cumsum(squared_S_stack) / np.sum(squared_S_stack)        #

r = 15
mdl.plot_svd(r,S_stack,Ecum_stack)                                      # Plot the singular value decay and cumulative energy

In [ ]:
# ===================================================================================================================
# Cumulative energy criteria to choose the reduced order
# ===================================================================================================================
tol = 0.0001                          # Energy tolerance: small threshold, denoting how much energy we can loose maximally
r_cum_stack = np.argmax(Ecum_stack > (1 - tol))  # Finds the first index where the condition is True. Returns 0 if the condition is never met (so check .any() if unsure)

print(f"Order of reduced model: {r_cum_stack}")
print(f"Energy captured by the snapshots: {100*Ecum_stack[r_cum_stack]}%")                 # Cumulative energy

In [ ]:
# ===================================================================================================================
# Choose the reduced order
# ===================================================================================================================
sys_order_stack = ???

In [ ]:
# ===================================================================================================================
# Data approximation
# ===================================================================================================================
Ur_stack = U_stack[:,:sys_order_stack]

data_approx_error = abs(Ur@(Ur.T@X_training[5]) - X_training[5])
mdl.visualize2d(t,
                data_approx_error,
                  contour=True,logscale=True, title = 'Data approximation error')

<font color = blue>**Your task**</font>

- <font color = blue>**choose the final projection basis and reduced order**</font>

In [ ]:
# ===================================================================================================================
# Choose the reduced order
# ===================================================================================================================
sys_order_final = ???
Ur_final        = ???
X_train_final   = ???

In [ ]:
# ===================================================================================================================
# Data compression
# ===================================================================================================================
X_reduced =  ???  # Project the X_train_final onto Ur_final!

<span style="color:deepskyblue; text-decoration:underline;">**Optimization problem**</span>

- In a nutshell, the learning consists of minimizing the following function:
$$
(\mathbf{S}^*, \mathbf{L}^*, \mathbf{K}^*) := \operatorname*{argmin}_{\mathbf{L},\mathbf{K}, \mathbf{S}} \left\| \dot{\widehat{\mathbf{X}}} - (\mathbf{S} - \mathbf{S}^T - \mathbf{L}^T \mathbf{L})\mathbf{K}^\top \mathbf{K} \widehat{\mathbf{X}} \right\|_F^2 + \mathcal{R}(\mathbf{L}, \mathbf{K}, \mathbf{S})
$$

<font color = blue>**Write the matrix** $\color{blue}{\hat{\mathbf{A}}}$ and $\color{blue}{\mathbf{J, R, Q}}$ **in terms of** $\color{blue}{\mathbf{S^*, L^*, K^*}}$. **Explain their properties.**</font>



 - We solve the optimization problem using stochastic gradient descent, implemented in Pytorch framework (https://docs.pytorch.org/tutorials/beginner/basics/intro.html). In this tutorial we will use the Adam optimizer (https://arxiv.org/pdf/1412.6980) , which is a very popular and efficient method for stochastic gradient-based optimization.
- The training process is programmed in the pre-defined function, available in `modules.py`. Here is the code snippet, explaining the inpt and output arguments:
    ```python
    def training(X_reduced, model, Params, lr_scheduler="cyclic", roll_out = False):

        """
        This is a training function.
        Inputs:
        X_reduced:    state data tensor
        model:        model, which defines unknown parameters and the forward rule, to get the prediction
        Params:       contains the relevant parameters such as epochs, learning rate values, and so on
        lr_scheduler: learning rate scheduler type, None for constant learning rate
        roll_out:     specifies if the time integrator is embedded


        Returns:
            trained model, loss_track, learning_rate_track
        """
    ```

- The training is performed by automatic differentiation, including the following steps:

    1. **Forward pass &  Loss calculation**

    Pass the data through the *model* to get a prediction. Calculate how wrong the prediction was (loss).

    2. **Zero the gradients**

    (`optimizer.zero_grad()`) - Clear old gradients before clculating new gradients (PyTorch specifics, it accumulates gradients from every batch, so you need to clear them.)

    3. **Backward pass**

    (`loss.backward()`) - Calculates the derivative of the loss w.r.t. every parameter, going backwards the computation graph. The gradients are stored in parameters, as attributes .grad .

    4. **Parameter update**

    (`optimizer.step()`) - updates the parameters values to reduce the loss according to the computed gradients.




In [ ]:
torch.manual_seed(23) # Fix the seed to get the same randoization

# ===================================================================================================================
# Model hypotheses
# ===================================================================================================================

class Model_linear(nn.Module):          # nn.Module - Base class for all neural network modules.
    """
    Here we define a linear model
                dx/dt = A x
    with no stability guarantees.
    """
    def __init__(self, sys_order):
        print("No Stability Gurantees!") # Initialization: initializae system components. In the __init__ method, runs once when the object is created.
        super().__init__()               # nn.Module initialization: an __init__() call to the parent class must be made before assignment on the child.

        scaler = 10
        self.A = nn.Parameter(torch.randn(sys_order, sys_order) # Initial guesses for the system matrix.
                                    / scaler
                                    )
    #

    #
    def forward(self, x):
        """Forward: "moves" the model forward, defines how are the outputs calculated.
           The outputs in our case are the derivatives.
           In the forward method runs every time you pass data through the model.
        """
        return x @ self.A.T # # Note: Using linear (x @ A.T) is equivalent to x^T A^T

<font color = blue>**Your task**</font>

- <font color= blue>**define the `model`, where the system hypothesis are formulated, with stability guarantee `Model_linear_stable` and without stbility guarantees `Model_linear`. Follow the advise comments in the code cell.**</font>

In [ ]:
class Model_linear_stable(nn.Module):
    """
    Here we define a linear model
                dx/dt = A x
    with stability guarantees: A = (J - R) Q
    """

    def __init__(self, sys_order):

        """
        Define model parameters.
            A = (J-R)Q,
        where J is skew-symmetric, R and Q are positive semi-definite matrics.
        This then guarantees stability of A.
        """
        print("Stability Gurantees!")

        super().__init__() # Call the initialization of the parent class (nn.Module)
        fac = sys_order

        # Initialize the parameters J,R, and Q , such that the matrix A = (J - R)Q is stable
        # The model updates parameters S , L , K
        self.S = nn.Parameter(torch.randn(sys_order, sys_order)
                                     / fac
                                     )
        self.L = nn.Parameter(torch.randn(sys_order, sys_order)
                                     / fac
                                     )
        self.K = nn.Parameter(torch.randn(sys_order, sys_order)
                                     / fac
                                     )

        J = ????? # skew-symmetric property
        R = ????? # symmetric positive definite property

        Q = ?????? # symmetric positive definite property
        A = ???????

        self.A = A

    def forward(self, x):
        """"Forward: "moves" the model forward, defines how are the outputs calculated.
           The outputs in our case are the derivatives.
           In the forward method runs every time you pass data through the model.

           x should be of dimension [n_init x n_dofs x n_times] ?
        """
        # Since the model updates parameters S, L, K, it is necessary to repeat the factorization in the forward method,
        # to enforce the correct update of the matrix A
        J = ????? # skew-symmetric property
        R = ????? # symmetric positive definite property

        Q = ?????? # symmetric positive definite property
        A = ???????

        self.A = A

        return x @ A.T

**Short note on decorators**

The next cell uses a *decorator* (https://book.pythontips.com/en/latest/decorators.html) to simplify the notation.

*What is a decorator? Here is a short example!*

```python

def decorator(func):
    def wrap():
        print("Some work before executing func()")

        func()

        print("Some work after executing func()")

    return wrap

@decorator
def function_to_decorate():
    print("I am THE function!")

#the @decorator is just a short way of saying:
function_to_decorate = decorator(function_to_decorate)

function_to_decorate()
#OUTPUT:  Some work before executing func()
#         I am THE function!
#         Some work after executing func()


```

We will use the `dataclass` decorator for our parameters, which automatically provides the special methods (such as `__init()__` , etc.)

In [ ]:
@dataclass
class parameters():
    #
    # Model Configuration parameters
    #
    stability_guarantee: bool = False # if true - a model with stability guarantees is used DELETE
    #
    # Training hyperparameters
    #
    bs: int = n_inits_train           # batch size
    num_epochs: int = 4000            # number of iterations
    weight_decay: float = 1e-8        #
    #
    # Data parameters
    #
    time_step: float = dt             # Time step. Note that it must be consistent with the used data set!
    num_inits: int = n_inits_train    # Number of initial conditions
    #
    # Learning rate parameters
    #
    step_size_up: int= 2000           # step for cyclic learning rate scheduler
    mode: str = "triangular2"         # cyclic scheduler mode
    cycle_momentum: bool = False      #
    base_lr: float = 1e-6             # basis learning rate for cyclic learning rate scheduler
    max_lr: float = 1e-2              # maximal learning rate for cyclic learning rate scheduler
    lr: float = 1e-2                  # constant learning rate

params = parameters()

In [ ]:
# ===================================================================================================================
# Training  without stability guqrantees
# ===================================================================================================================

model_nostab = Model_linear(sys_order=sys_order_final).double()

model_nostab, loss_track_nostab, lr_track_nostab = mdl.training(X_reduced, model_nostab, params)
A_learn_nostab = model_nostab.A.detach().numpy()

In [ ]:
# ===================================================================================================================
# Training  with stability guqrantees
# ===================================================================================================================
model_stab = ????                                    # Define the model and solve the problem analogously to model_nostab!
model_stab, loss_track_stab, lr_track_stab = ????
A_learn_stab = model_stab.A.detach().numpy()

### Validation of the Reduced Model

- Finally, we analyze the performance of the identified reduced model. We simulate the reduced model using initial conditions projected from the available data sets:
$$
\dot{\widehat{\mathbf{x}}}(t) = \hat{\mathbf{A}} \widehat{\mathbf{x}}(t), \quad \widehat{\mathbf{x}}(0) = \mathbf{U}_r^\top \mathbf{x}(0)
$$

- The model can be analyzed in the following way:

    **1. Reconstruction**

    To compare the reduced trajectories $\widehat{x}$ with the original high-dimensional data, we reconstruct the high-dimensional representation $\mathbf{x}_{\text{rec}}$:

    $$
    \mathbf{x}_{\text{rec}}(t) = \mathbf{U}_r \widehat{\mathbf{x}}(t)
    $$

    **2. Error Analysis**

    The relative reconstruction error $\epsilon$ is calculated to quantify the accuracy of the approximation:

    $$
    \epsilon = \frac{\| \mathbf{x}(t) - \mathbf{x}_{\text{rec}}(t) \|}{\| \mathbf{x}(t) \|}
    $$

    **3. Stability Check**

    Furthermore, we verify the stability of the system by examining the eigenvalues of the reduced system matrix $\widehat{A}$. Stability is confirmed if the real parts of all eigenvalues are strictly negative:

    $$
    \text{Re}(\lambda_i(\hat{\mathbf{A}})) < 0 \quad \forall i
    $$

<font color = blue>**Your task**</font>

- <font color = blue>**plot the eigenvalues of the `A_stab` and `A_nostab`** </font>
- <font color = blue>**Simulate both reduced models for different initial conditions, contained in the given data sets, including the test data set, which was not involved in the training process. Compare the performance of the model with guaranteed stability and without.**</font>

In [ ]:
# ===================================================================================================================
# Eigenvalues
# ===================================================================================================================

[eigs_stab, vec_stab] = ????      # Eigenvalues of the matrix with stability guarantees (use np.linalg.eig)
[eigs_nostab, vec_nostab] = ????  # Eigenvalues of the matrix with NO stability guarantees

mdl.plot_eigs(eigs_nostab, ev_label = 'No stability', ev2 = eigs_stab,ev2_label = 'Stability',xlim = [-10,10])

In [ ]:
# ===================================================================================================================
# Reduced-order simulation
# ===================================================================================================================
time  = np.squeeze(t)
def linear_model(t, x, A_learn): return A_learn @ x # Define linear system as a function

sol_original = X_test[2, :, :]                      # Choose the test data set

x0 = Ur_final.T @ sol_original[ :, 0]               # Define initial condition from the chosen data set

#
# Stable ROM
#
sol_red_stab = solve_ivp(
        linear_model,
        [t[0], t[-1]],
        x0,
        args=(A_learn_stab,),
        t_eval=time,
        atol=1e-8,
        rtol=1e-8,
    )
sol_full_stab = Ur_final @ sol_red_stab.y

#opinf_error_stab = np.linalg.norm(sol_original - sol_full_stab, axis = 0)/np.linalg.norm(sol_original, axis = 0) # Error in time
opinf_error_stab = abs(sol_original - sol_full_stab)/np.linalg.norm(sol_original, axis = 0) # Error in time

#
# "Unstable" ROM
#
sol_red_nostab = solve_ivp(
        linear_model,
        [t[0], t[-1]],
        x0,
        args=(A_learn_nostab,),
        t_eval=time,
        atol=1e-8,
        rtol=1e-8,
    )
sol_full_nostab = Ur_final @ sol_red_nostab.y

# opinf_error_nostab = np.linalg.norm(sol_original - sol_full_nostab, axis = 0)/np.linalg.norm(sol_original, axis = 0) # Error in time
opinf_error_nostab = abs(sol_original - sol_full_nostab)/np.linalg.norm(sol_original, axis = 0) # Error in time


In [ ]:
mdl.visualize3d(t, sol_original, title="Test Initial condition. \n Full-order model")
mdl.visualize3d(t, sol_full_stab, title="Test Initial condition. \n Reduced-order model with stability guarantees.")
mdl.visualize3d(t, sol_full_nostab, title="Test Initial condition. \n Reduced-order model, no stability guarantees.")

In [ ]:
#To do: make uniform color grid for error plots.
mdl.visualize2d(t, opinf_error_stab,
                  contour=True,logscale=True, title = 'Relative error: ROM with stability guarantees')
mdl.visualize2d(t, opinf_error_nostab,
                  contour=True,logscale=True, title = 'Relative error: ROM no stability guarantees')

In [ ]:
#mdl.visualize1d( [0.1, 0.9, 0.5], dt, sol_original, title="Initial condition 1", red_snapshots = sol_full_stab)

<font color = blue>**Your task:**</font>

- <font color = blue>**change different hyperparaneters, such as epoch number, or learning rates. How does it affect the loss?**</font>
- <font color = blue>**play with different parameters for reduction - reduced order, global or local basis - can you find the best setting to approximate the test data set?**</font>
- <font color = blue>**can you identify the issues of the current optimization problem formulation? How can you improve it (data scaling/centering, derivative approximation ...) ?**</font>

---

## Part 2: Nonlinear Systems

---

### Local vs global asymptotic stability

<span style="color:deepskyblue; text-decoration:underline;">**Definitions**</span>

- Recall that up to now we cansidered *local asymptotic stability (L.A.S.)*: if there exist $\delta > 0$ such that $\|\mathbf{x}(0) - x_e \| < \delta \; \Rightarrow \;  \mathbf{x}(t) \rightarrow x_e$ as $t \to \infty.$
<br>It means, the system is stable **near or at** $x_e$.

- Different concept is *global asymptotic stability (G.A.S.)*: if **for every** $\mathbf{x} (t)$ we have $\mathbf{x} (t) \rightarrow x_e$ as $t \rightarrow \infty$.<br>It means, at any initial condition the system will converge.

- For **linear systems**, such as $ \dot{\mathbf{x}} (t) = \mathbf{A}\mathbf{x}(t) , \; \mathbf{x}(0) = \mathbf{x}_0$ : $\boxed{\text{L.A.S.} \Rightarrow \text{G.A.S.}}$


<span style="color:deepskyblue; text-decoration:underline;">**L.A.S. quadratic system**</span>

- Next, we consider a quadratic system

$$\dot{\mathbf{x}}(t) = \mathbf{A} \mathbf{x}(t) + \mathbf{H} (\mathbf{x}(t) \otimes \mathbf{x}(t)) + \mathbf{B}$$

- We can infer a L.A.S. nonlinear system by solving the folowing regression problem:
$$
\mathbf{(S^*, L^*, K^*, H^*, B)} := \operatorname*{argmin}_{\mathbf{L,K,S,H}} \left\| \dot{\mathbf{X}} - \left[ (\mathbf{S} - \mathbf{S}^\top - \mathbf{L}^\top \mathbf{L})\mathbf{K}^\top \mathbf{K} \quad  \mathbf{H} \quad \mathbf{B} \right] \begin{bmatrix} \mathbf{X} \\ \mathbf{X} \otimes \mathbf{X} \\ \boldsymbol{1} \end{bmatrix} \right\|_F^2 + \mathcal{R}(\mathbf{S,L,K,H,B})
$$

- To identify a G.A.S. system , more assumptions are needed.


**G.A.S. quadratic system**

- We can formulate constraints for quadratic G.A.S. systems, if the considered system is **generalized energy-preserving** , which means $\mathbf{x}^\top \mathbf{H} (\mathbf{x} \otimes \mathbf{x}) = 0$ for all $\mathbf{x}$.

- A quadratic system is **generalized energy-preserving** w.r.t. $\mathbf{Q}$ if $\mathbf{H} = [\mathbf{H}_1 \mathbf{Q} , \ldots , \mathbf{H}_n \mathbf{Q}]$, where $\mathbf{H}_j = -\mathbf{H}_j^\top, j = 1, \ldots, n$, and global Lyapunov function is defined as $\frac{1}{2} \mathbf{x}^\top \mathbf{Q} \mathbf{x}$.

- For generalized energy-preserving quadratic system, the G.A.S. is guaranteed, if the symmetric part of A is asymptotically stable. it means, we can identify the G.A.S. system by solving the following regression problem:$$
(\mathbf{A^*, H^*, B}) := \operatorname*{argmin}_{\mathbf{A,H}} \left\| \dot{\mathbf{X}} - \left[ \mathbf{A} \quad  \mathbf{H} \quad \mathbf{B}\right] \begin{bmatrix} \mathbf{X} \\ \mathbf{X} \otimes \mathbf{X} \\ \boldsymbol{1} \end{bmatrix} \right\|_F^2 + \mathcal{R}(\mathbf{S,L,K,H, B}) $$
with the stability constraints
$$\mathbf{A} = (\mathbf{S} - \mathbf{S}^\top - \mathbf{L}^\top \mathbf{L})\mathbf{K}^\top \mathbf{K} $$
$$\mathbf{H} = [\mathbf{H}_1, \ldots, \mathbf{H}_n] , \quad \text{with} \quad \mathbf{H}_j = -\mathbf{H}_j^\top \quad \text{and} \quad \mathbf{Q} = \mathbf{I}
$$





In [ ]:

# data = loadmat("./data/Burgers_dirichilet_data.mat")

## For colab
## Download the specific file to the current directory
!wget  -O data_nnl.mat https://github.com/goyalpike/quadratic-stable-opinf/raw/refs/heads/main/Examples/data/Burgers_dirichilet_data.mat

## to delete a file execute: !rm filename.py



In [ ]:
data = loadmat("data_nnl.mat")
t     = data["t"].T                                           # Time vector [n_times x 1]
X_all = data["X_data"].transpose(0, 2, 1)                     # Snapshots [n_inits x n_dofs x n_times]

time  = np.squeeze(t)
n_inits, n_dofs, n_times,  = X_all.shape                      # Assign the repective dimensions: number of initial conditions, number of time points, number of DOFs

dt = (t[2] - t[1]).item()                                     # Calculate the time step from the time vector

# ====================================================================================
# Training data
# ====================================================================================

all_idxs   = list(np.arange(0, n_inits))           # List of all indices
test_idxs  = list([4, 8, 12])                      # Data sets for test (e.g., data set corresponding to initial condition 4, to initial condition 8, and 12)
train_idxs = list(set(all_idxs) - set(test_idxs))  # Training indices

X_test     = X_all[test_idxs]                      # Include the data sets with indices test_idxs to the test set
X_training = X_all[train_idxs]                     # Include the data sets with indices training_idxs to the training set

n_inits_train = X_training.shape[0]                # Number of the training indices

print(f"Training trajectories: {X_training.shape[0]}")
print(f"Test trajectories: {X_test.shape[0]}")

# ====================================================================================
# Data compression
# ====================================================================================

X_stack = X_training.transpose(1, 0, 2).reshape(X_training.shape[1], -1) # Stack the data set for each initial condition into one array

U_stack,S_stack,W_stack = np.linalg.svd(X_stack, full_matrices=False)    # Left singular vecs, singular vals, right singular vecs

squared_S_stack = S_stack**2                                             # Cumulative energy
Ecum_stack = np.cumsum(squared_S_stack) / np.sum(squared_S_stack)                    #

#
# Cumulative energy criteria to choose the reduced order
#
tol = 0.0001                          # Energy tolerance: small threshold, denoting how much energy we can loose maximally
sys_order_final = np.argmax(Ecum_stack > (1 - tol))  # Finds the first index where the condition is True. Returns 0 if the condition is never met (so check .any() if unsure)

print(f"Order of reduced model: {sys_order_final}")
print(f"Energy captured by the snapshots: {100*Ecum_stack[sys_order_final]}%")                 # Cumulative energy

Ur_final        = U_stack[:,:sys_order_final] #Vr_stack
X_train_final   = X_stack

X_reduced = Ur_final.T@X_train_final

mdl.plot_svd(sys_order_final,S_stack,Ecum_stack)                                      # Plot the singular value decay and cumulative energy

In [ ]:

class Model_LAS(nn.Module):
    def __init__(self, sys_order, B_term=True, *arg, **kwargs):
        super().__init__()
        """
            Define model parameters.
            A = (J-R)Q,
            where J is skew-symmetric, R and Q are positive semi-definite matrics.
            This then guarantees stability of A.
        """
        print("Local Stability Gurantees!")
        self.n = sys_order
        factor = sys_order
        self.B_term = B_term
        self.S = torch.nn.Parameter(torch.randn(sys_order, sys_order) / factor)
        self.L = torch.nn.Parameter(torch.randn(sys_order, sys_order) / factor)
        self.K = torch.nn.Parameter(torch.randn(sys_order, sys_order) / factor)

        self.B = torch.nn.Parameter(torch.zeros(sys_order, 1))
        self.H = torch.nn.Parameter(torch.zeros(sys_order, sys_order**2))


    @property
    def A(self):
        """Construct stable matrix A."""
        J = self.S - self.S.T
        R = self.L @ self.L.T

        Q = self.K @ self.K.T
        return (J - R) @ Q

    def forward(self, x):
        """Create a forward pass defined right-hand-side of quadratic systems."""
        x_A = x @ self.A.T
        H_3d = self.H.view(self.n,self.n,self.n)
        x_H = torch.einsum('oij, mti, mtj -> mto', H_3d, x, x) # H(o, a, b) * X(m, a, k) * X(m, b, k) -> Output(m, k, o)
        if self.B_term:
            return x_A + x_H + self.B.T
        else:
            return x_A + x_H

class Model_GAS(nn.Module):
    def __init__(self, sys_order, B_term=True, *arg, **kwargs):
        super().__init__()
        """
            Define model parameters.
            A = (J-R)Q,
            where J is skew-symmetric,
                  R and Q are positive semi-definite matrices.
            We have assumed Q to be identity (Q = I); hence, it is removed from the optimzer.
            This then guarantees global asymptotic stability of A.
        """
        print("Global Stability Gurantees!")
        factor = 10
        self.B_term = B_term
        self.sys_order = sys_order                                                 # System dimension
        self.S = torch.nn.Parameter(torch.randn(sys_order, sys_order) / factor)    # Factor, corresponding to skew-symmetric J
        self.L = torch.nn.Parameter(torch.randn(sys_order, sys_order) / factor)    # Factor, corresponding to symmetric positive definite R
        if B_term:
            self.B = torch.nn.Parameter(torch.zeros(sys_order, 1))                 # Input term
        self.HQ = torch.nn.Parameter(torch.zeros(sys_order, sys_order, sys_order) / factor) # Factor, corresponding to energy-preserving H

    @property
    def A(self):
        """Construct stable matrix A with Q being identity."""
        J = self.S - self.S.T
        R = self.L @ self.L.T
        return J - R

    @property
    def H(self):
        """Constuct energy preserving quadratic operator."""
        HQ_T = self.HQ.permute(0, 2, 1)  # Transpose the factor HQ - HQ^T
        H_permuted = (self.HQ - HQ_T).permute(1, 0, 2)
        return H_permuted.contiguous().reshape(self.sys_order, self.sys_order**2) # .contiguous() forces PyTorch to make a copy.

    def forward(self, x):
        """Create a forward pass defined right-hand-side of quadratic systems."""
        x_A = x @ self.A.T
        H_3d = self.H.view(self.sys_order,self.sys_order,self.sys_order)
        x_H = torch.einsum('oij, mti, mtj -> mto', H_3d, x, x) # H(o, a, b) * X(m, a, k) * X(m, b, k) -> Output(m, k, o)
        if self.B_term:
            return x_A + x_H + self.B.T
        else:
            return x_A + x_H

In [ ]:
@dataclass
class parameters():
    #
    # Model Configuration parameters
    #
    #
    # Training hyperparameters
    #
    bs: int = n_inits_train           # batch size
    num_epochs: int = 4000            # number of iterations
    weight_decay: float = 1e-8        #
    #
    # Data parameters
    #
    time_step: float = dt             # Time step. Note that it must be consistent with the used data set!
    num_inits: int = n_inits_train    # Number of initial conditions
    #
    # Learning rate parameters
    #
    step_size_up: int= 2000           # step for cyclic learning rate scheduler
    mode: str = "triangular2"         # cyclic scheduler mode
    cycle_momentum: bool = False      #
    base_lr: float = 1e-6             # basis learning rate for cyclic learning rate scheduler
    max_lr: float = 1e-2              # maximal learning rate for cyclic learning rate scheduler
    lr: float = 1e-2                  # constant learning rate

params = parameters()

In [ ]:

# ===================================================================================================================
# Training  L.A.S.
# ===================================================================================================================
model_las = Model_LAS(sys_order=sys_order_final).double()

model_las, loss_track_las, lr_track_las = mdl.training(X_reduced, model_las, params, lr_scheduler="cyclic", roll_out = False)
A_LAS = model_las.A.detach().numpy()
H_LAS = model_las.H.detach().numpy()
B_LAS = (model_las.B.detach().numpy().reshape(-1,))


In [ ]:

# ===================================================================================================================
# Training  G.A.S.
# ===================================================================================================================

model_gas = Model_GAS(sys_order=sys_order_final).double()

model_gas, loss_track_gas, lr_track_gas = mdl.training(X_reduced, model_gas, params, lr_scheduler="cyclic", roll_out = False)
A_GAS = model_gas.A.detach().numpy()
H_GAS = model_gas.H.detach().numpy()
B_GAS = (model_gas.B.detach().numpy().reshape(-1,))




<font color = blue>**Your task:**</font>

- <font color = blue>**find for which initial conditions the L.A.S. solution blows up?**</font>

In [ ]:
def model_quad_OpInf(t, x, A, H, B):
    """Define a vector field of a quadratic system."""
    return A @ x + H @ np.kron(x, x) + B

sol_original = X_test[2, :, :]                      # Choose the test data set

x0 = Ur_final.T @ sol_original[:,0]
sol_LAS_red = solve_ivp(model_quad_OpInf, [t[0], t[-1]], x0,args=(A_LAS, H_LAS, B_LAS), t_eval=time)
sol_LAS = Ur_final@ sol_LAS_red.y

nlas = sol_LAS.shape[1]
if sol_LAS.shape[1] != t.shape[0]:
    print("LAS solution has blown up.")


sol_GAS_red = solve_ivp(model_quad_OpInf, [t[0], t[-1]], x0,args=(A_GAS, H_GAS, B_GAS), t_eval=time)
sol_GAS = Ur_final@ sol_GAS_red.y

In [ ]:
mdl.visualize3d(t, sol_original, title="Test Initial condition. \n Full-order model")
mdl.visualize3d(t[:nlas], sol_LAS, title="Test Initial condition. \n LAS model")
mdl.visualize3d(t, sol_GAS, title="Test Initial condition. \n GAS model")